# 05 — Rio de la Plata / Guarani, Google & TESSERA Embeddings (CPU-only)

Delineates agricultural fields in the Rio de la Plata basin using pre-computed satellite embeddings (Google 64-D and TESSERA 128-D) with unsupervised K-means clustering. **No GPU required.**

The study area is loaded directly from a **GEE vector asset** rather than a local file, demonstrating agribound's GEE asset input support.

**Estimated runtime:** ~10–20 minutes (5 years, CPU only).

**Prerequisites:**
```bash
pip install agribound[gee,tessera]
agribound auth --project YOUR_GEE_PROJECT
```

## Configuration

In [ ]:
from pathlib import Path

import agribound

GEE_PROJECT = "YOUR_GEE_PROJECT"  # <-- Replace with your GEE project ID

OUTPUT_DIR = Path("../../outputs/riodelaplata")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Study area as a GEE vector asset (no local file needed)
STUDY_AREA = "projects/ssebop-471916/assets/riodelaplata_guarani"

YEARS = range(2020, 2025)

## Part 1: Google Satellite Embeddings (64-D)

In [ ]:
all_results = {}

for year in YEARS:
    print(f"Processing {year} (Google embeddings)...", end=" ")
    output_path = OUTPUT_DIR / f"fields_google_{year}.gpkg"

    gdf = agribound.delineate(
        study_area=STUDY_AREA,
        source="google-embedding",
        year=year,
        engine="embedding",
        output_path=str(output_path),
        gee_project=GEE_PROJECT,
        device="cpu",
        min_area=5000,
    )
    all_results[f"google_{year}"] = gdf
    print(f"{len(gdf)} fields")

## Part 2: TESSERA Embeddings (128-D)

In [ ]:
for year in YEARS:
    print(f"Processing {year} (TESSERA embeddings)...", end=" ")
    output_path = OUTPUT_DIR / f"fields_tessera_{year}.gpkg"

    gdf = agribound.delineate(
        study_area=STUDY_AREA,
        source="tessera-embedding",
        year=year,
        engine="embedding",
        output_path=str(output_path),
        gee_project=GEE_PROJECT,
        device="cpu",
        min_area=5000,
    )
    all_results[f"tessera_{year}"] = gdf
    print(f"{len(gdf)} fields")

## Summary

In [ ]:
print(f"{'Source':<20} {'Year':<6} {'Fields':>6} {'Area (ha)':>12}")
print(f"{'-' * 20} {'-' * 6} {'-' * 6} {'-' * 12}")
for key, gdf in sorted(all_results.items()):
    area_ha = gdf["metrics:area"].sum() / 10000 if "metrics:area" in gdf.columns else 0
    source, year = key.rsplit("_", 1)
    print(f"{source:<20} {year:<6} {len(gdf):>6} {area_ha:>12,.1f}")

## Visualization: Google vs TESSERA Comparison

In [ ]:
from agribound.visualize import show_comparison

latest = max(YEARS)
google_gdf = all_results.get(f"google_{latest}")
tessera_gdf = all_results.get(f"tessera_{latest}")

if google_gdf is not None and tessera_gdf is not None:
    m = show_comparison(
        [google_gdf, tessera_gdf],
        labels=[f"Google {latest}", f"TESSERA {latest}"],
        basemap="Esri.WorldImagery",
        output_html=str(OUTPUT_DIR / "map_comparison.html"),
    )
    m